# Recomendación de amistad en redes sociales

Este es un ejemplo (simulado) utilizando PyTorch Geometric (PyG), simulando un grafo homogéneo **no bipartito** y entrenando un modelo GraphSAGE para resolver la tarea mediante la evaluación de pares de nodos positivos (enlaces reales) y negativos (enlaces inexistentes).
<br>
El objetivo es predecir si debería existir una arista (posible amistad) entre dos nodos (usuarios)


In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.2 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.data import Data
from torch_geometric.utils import negative_sampling
import torch_geometric.transforms as T
from torch_geometric.nn import SAGEConv

# 1. SIMULACIÓN DEL GRAFO (Detección de Fraude)
# Supongamos 1,000 cuentas bancarias (nodos) con 16 características cada una
num_nodes = 1000
num_features = 16
x = torch.randn(num_nodes, num_features) # Características

# Simulamos 5,000 amistades/enlaces existentes
# Generamos índices de origen y destino aleatorios
edge_index = torch.randint(0, num_nodes, (2, 5000), dtype=torch.long)

# Eliminamos auto-bucles para mantener consistencia
mask = edge_index[0] != edge_index[1]
edge_index = edge_index[:, mask]

data = Data(x=x, edge_index=edge_index)

# 2. DIVISIÓN DE ENLACES (Train, Val, Test)
# El transformador divide de forma automática las aristas en positivas y negativas
transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.2,
    is_undirected=True,
    add_negative_train_samples=True # Genera ejemplos negativos para el entrenamiento
)
train_data, val_data, test_data = transform(data)

print(f"Enlaces de entrenamiento positivos: {train_data.edge_label_index.size(1)//2}")
print(f"Enlaces de validación positivos: {val_data.edge_label_index.size(1)//2}")
print(f"Enlaces de prueba positivos: {test_data.edge_label_index.size(1)//2}")

Enlaces de entrenamiento positivos: 1757
Enlaces de validación positivos: 250
Enlaces de prueba positivos: 501



# Arquitectura del Modelo GNN (GraphSAGE)

Para generar las variables de un enlace, el modelo la obtiene combinando las de sus nodos.
1. Codificador (GNN): Genera representaciones vectoriales (embeddings) para cada nodo integrando la información de sus vecinos.
2. Decodificador (Dot Product):  Producto punto de los embeddings de cada par de nodos para predecir la probabilidad de que exista un enlace.

In [ ]:
class Net(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super().__init__()
        # Codificador basado en GraphSAGE
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

    def decode(self, z, edge_label_index):
        # Producto punto entre los nodos origen y destino para predecir el enlace
        return (z[edge_label_index[0]] * z[edge_label_index[1]]).sum(dim=-1)

    def decode_all(self, z):
        # Utilidad para evaluar la matriz completa si fuera necesario
        prob_adj = z @ z.t()
        return prob_adj

El modelo busca minimizar la entropía de la salida (aristas verdaderas con 1, falsas con 0)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Net(num_features, 64).to(device)
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.01)
criterion = torch.nn.BCEWithLogitsLoss()

def train():
    model.train()
    optimizer.zero_grad()

    # 1. Codificar nodos usando la estructura de entrenamiento
    z = model.encode(train_data.x.to(device), train_data.edge_index.to(device))

    # 2. Decodificar las etiquetas de los enlaces (positivos y negativos de entrenamiento)
    out = model.decode(z, train_data.edge_label_index.to(device))

    # 3. Calcular la pérdida
    loss = criterion(out, train_data.edge_label.to(device))
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def test(data_split):
    model.eval()
    z = model.encode(data_split.x.to(device), data_split.edge_index.to(device))
    out = model.decode(z, data_split.edge_label_index.to(device)).view(-1).sigmoid()

    # Calculamos el ROC-AUC para medir el rendimiento de clasificación
    from sklearn.metrics import roc_auc_score
    y_true = data_split.edge_label.cpu().numpy()
    y_pred = out.cpu().numpy()
    return roc_auc_score(y_true, y_pred)

# Bucle principal de ejecución
for epoch in range(1, 41):
    loss = train()
    val_auc = test(val_data)
    print(f'Epoch: {epoch:02d}, Loss: {loss:.4f}, Val AUC: {val_auc:.4f}')

test_auc = test(test_data)
print(f'\n▶ ROC-AUC Final en el conjunto de Prueba: {test_auc:.4f}')

Epoch: 01, Loss: 1.5907, Val AUC: 0.4821
Epoch: 02, Loss: 1.4501, Val AUC: 0.4704
Epoch: 03, Loss: 1.0102, Val AUC: 0.4677
Epoch: 04, Loss: 1.0138, Val AUC: 0.4782
Epoch: 05, Loss: 1.0488, Val AUC: 0.4932
Epoch: 06, Loss: 0.9129, Val AUC: 0.5064
Epoch: 07, Loss: 0.7674, Val AUC: 0.5139
Epoch: 08, Loss: 0.6992, Val AUC: 0.5117
Epoch: 09, Loss: 0.6774, Val AUC: 0.5039
Epoch: 10, Loss: 0.6766, Val AUC: 0.4957
Epoch: 11, Loss: 0.6845, Val AUC: 0.4875
Epoch: 12, Loss: 0.6886, Val AUC: 0.4850
Epoch: 13, Loss: 0.6813, Val AUC: 0.4852
Epoch: 14, Loss: 0.6638, Val AUC: 0.4869
Epoch: 15, Loss: 0.6422, Val AUC: 0.4918
Epoch: 16, Loss: 0.6221, Val AUC: 0.4953
Epoch: 17, Loss: 0.6055, Val AUC: 0.4980
Epoch: 18, Loss: 0.5923, Val AUC: 0.5010
Epoch: 19, Loss: 0.5816, Val AUC: 0.5025
Epoch: 20, Loss: 0.5724, Val AUC: 0.5043
Epoch: 21, Loss: 0.5643, Val AUC: 0.5052
Epoch: 22, Loss: 0.5567, Val AUC: 0.5063
Epoch: 23, Loss: 0.5496, Val AUC: 0.5061
Epoch: 24, Loss: 0.5426, Val AUC: 0.5073
Epoch: 25, Loss:

# Clasificación de aristas

Si en cambio se busca clasificar el tipo de conexión que existe entre dos nodos, el modelo a seguir cambia:
<br>
1. Las capas convolucionales para el grafo generan embeddings para cada nodo.
2. Se define el embedding de una arista por concatenación, y
3. Se clasifica las aristas

* Si las aristas tienen atributos, estos se anaden al embedding del paso 2

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

# 1. SIMULACIÓN DE DATOS (Nodos y Aristas con Clases)
num_nodes = 1000
num_node_features = 16
num_classes_edge = 3  # Ej: 0 = Amistad, 1 = Profesional, 2 = Odio

x = torch.randn(num_nodes, num_node_features)
edge_index = torch.randint(0, num_nodes, (2, 4000), dtype=torch.long)

# Simulamos etiquetas para cada arista (las clases que queremos predecir)
edge_y = torch.randint(0, num_classes_edge, (4000,), dtype=torch.long)

# Máscaras simples para entrenamiento y prueba de las aristas
edge_train_mask = torch.zeros(4000, dtype=torch.bool)
edge_train_mask[:3000] = True
edge_test_mask = ~edge_train_mask

data = Data(x=x, edge_index=edge_index, edge_y=edge_y)

# 2. MODELO PARA CLASIFICACIÓN DE ARISTAS
class EdgeClassifierNet(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, num_edge_classes):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)

        # El clasificador final toma el embedding de los DOS nodos que forman la arista
        # Al concatenar el nodo origen y destino, la entrada tiene tamaño: hidden_channels * 2
        self.edge_classifier = Sequential(
            Linear(hidden_channels * 2, hidden_channels),
            ReLU(),
            Linear(hidden_channels, num_edge_classes)
        )

    def forward(self, x, edge_index):
        # 1. Generar embeddings para todos los nodos
        z = self.conv1(x, edge_index).relu()
        z = self.conv2(z, edge_index)

        # 2. Obtener los nodos origen (u) y destino (v) de cada arista
        row, col = edge_index[0], edge_index[1]
        z_src = z[row]
        z_dst = z[col]

        # edge_features = torch.cat([z_src, z_dst, data.edge_attr], dim=-1)

        # 3. Concatenar las representaciones de ambos nodos para clasificar la arista
        edge_features = torch.cat([z_src, z_dst], dim=-1)
        return self.edge_classifier(edge_features)

# 3. PIPELINE DE ENTRENAMIENTO
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EdgeClassifierNet(num_node_features, 64, num_classes_edge).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

# Ejecución de una época de entrenamiento
model.train()
optimizer.zero_grad()
out = model(data.x.to(device), data.edge_index.to(device))

# Calculamos la pérdida solo en las aristas destinadas a entrenamiento
loss = criterion(out[edge_train_mask], data.edge_y[edge_train_mask].to(device))
loss.backward()
optimizer.step()

print(f"Entrenamiento completado. Pérdida de la época: {loss.item():.4f}")

Entrenamiento completado. Pérdida de la época: 1.0998


# Ejercicio

1. Obtener las estadísticas relevantes y las gráficas de evolución de la precisión
2. Modificar el código para incluir atributos de las aristas